# Legacy / Initial approach - based on Hanna’s Medium article

This notebook reproduces the original approach from the Medium article by Hanna Pylieva: [“How to visualize image feature vectors”](https://hanna-shares.medium.com/how-to-visualize-image-feature-vectors-1e309d45f28f).

It works for flat folders with a few images, but breaks for:
 - nested datasets (train/test/validation)
 - large datasets
 - reproducibility across machines
 - consistent ordering between embeddings, metadata and sprites

These limitations motivated the refactoring into the scripts/ pipeline used in this repository.

Note - make sure data is donwloaded and prepared properly before running the below code.


## Extract feature vectors

### Problems with this approach

 - relies on os.walk order (non-deterministic)
 - regex parsing paths (re.findall) is fragile
 - assumes fixed folder depth ([3:5])
 - no control over image count
 - no filename normalization
 - sprite/metadata/embeddings can silently go out of sync
 - forward hooks for embeddings (hacky, outdated)
 - hardcoded ImageNet normalization
 - cannot scale to multiple datasets


In [ ]:
import os
import re
import csv
from PIL import Image
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms

def get_vector(input_image):
    image = input_image.convert("RGB")  # in case input image is not in RGB format
    img_t = transform(image)
    batch_t = torch.unsqueeze(img_t, 0)
    my_embedding = torch.zeros([1, 512, 1, 1])
    def copy_data(m, i, o):
        my_embedding.copy_(o.data)
    h = layer.register_forward_hook(copy_data)
    model(batch_t)
    h.remove()
    return my_embedding.squeeze().cpu().numpy()

model = models.resnet18(pretrained=True)
layer = model._modules.get('avgpool')
model.eval()
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )])
im_path = '../images'
im_names = [os.path.join(root, name)
            for root, dirs, files in os.walk(im_path)
            for name in files
            if name.endswith(".jpeg")]


# This assumes fixed folder depth and will break for real datasets.
existing_images_df = pd.DataFrame([re.findall(r"[\w']+", im_name)[3:5] for im_name in im_names],
                                  columns=['cat_id', 'pid'])
existing_images_df['impath'] = im_names
vecs = [list(get_vector(Image.open(impath))) for _, pid, impath in existing_images_df.values]
with open('../vis/feature_vecs.tsv', 'w') as fw:
    csv_writer = csv.writer(fw, delimiter='\t')
    csv_writer.writerows(vecs)

In [22]:
re.findall(r"[\w']+", '../images/animals_dataset/test/cat/cat_13.jpeg'), re.findall(r"[\w']+", '../images/animals_dataset/test/cat/cat_13.jpeg')[3:5]

(['images', 'animals_dataset', 'test', 'cat', 'cat_13', 'jpeg'],
 ['cat', 'cat_13'])

In [23]:
existing_images_df.head()

,cat_id,pid,impath
0,cat,cat_13,../images/animals_dataset/test/cat/cat_13.jpeg
1,cat,cat_6,../images/animals_dataset/test/cat/cat_6.jpeg
2,cat,cat_7,../images/animals_dataset/test/cat/cat_7.jpeg
3,cat,cat_12,../images/animals_dataset/test/cat/cat_12.jpeg
4,cat,cat_15,../images/animals_dataset/test/cat/cat_15.jpeg


## Create a sprite image

In [24]:
import numpy as np

images = [Image.open(filename).resize((100,100)) for filename in existing_images_df['impath']]
image_width, image_height = images[0].size
one_square_size = int(np.ceil(np.sqrt(len(images))))
master_width = (image_width * one_square_size)
master_height = image_height * one_square_size
spriteimage = Image.new(
    mode='RGBA',
    size=(master_width, master_height),
    color=(0,0,0,0))  # fully transparent
for count, image in enumerate(images):
    div, mod = divmod(count,one_square_size)
    h_loc = image_width*div
    w_loc = image_width*mod
    spriteimage.paste(image,(w_loc,h_loc))
spriteimage.convert("RGB").save('../vis/sprite.jpg', transparency=0)

## Create a metadata file

In [25]:
metadata = existing_images_df[['cat_id', 'pid']].to_csv('../vis/metadata.tsv', sep='\t', index=False)

## Create a config file

modify `vis/projector_config.pbtxt`:

```
embeddings {
  tensor_path: "feature_vecs.tsv"
  metadata_path: "metadata.tsv"
  sprite {
    image_path: "sprite.jpg"
    single_image_dim: [100, 100]
  }
}
```

## Run tensorboard with the folder containing the config file.

```
tensorboard --logdir ./vis
```